In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 没有记忆

#### 如果没有记忆，看看agent 执行的结果。

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os



# model = init_chat_model("deepseek-chat")
# Alternative: local model via Ollama
# model = init_chat_model("qwen2.5:14b", model_provider="ollama")
model = init_chat_model(
    model="agnes-2.0-flash",
    model_provider="openai",
    base_url=os.getenv("AGNES_BASE_URL"),
    api_key=os.getenv("AGNES_API_KEY"),
)


agent = create_agent(
    model
)

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content="你好，我是卫平，我喜欢蓝色")

response = agent.invoke(
    {"messages": [question]} 
)

In [4]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='你好，我是卫平，我喜欢蓝色', additional_kwargs={}, response_metadata={}, id='e3bae7ba-81ac-43cb-8122-67d84266cf84'),
              AIMessage(content='你好，卫平！很高兴认识你。蓝色是一种很宁静、深邃的颜色，很多人都会喜欢它。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 217, 'total_tokens': 240, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'agnes-2.0-flash', 'system_fingerprint': None, 'id': 'chatcmpl-3377cdad-9e08-4e94-9e3f-6707fc508ce1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ebc3e-1d4a-7452-a73f-3a47e729930d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 217, 'output_tokens': 23, 'total_tokens': 240, 'input_token_details': {}, 'output_token_details': {}})]}


In [5]:
question = HumanMessage(content="我喜欢什么颜色？")

response = agent.invoke(
    {"messages": [question]} 
)

pprint(response)

{'messages': [HumanMessage(content='我喜欢什么颜色？', additional_kwargs={}, response_metadata={}, id='f313f356-03f6-4611-a0e5-bafae713f21f'),
              AIMessage(content='我不确定。作为一个人工智能，我无法知道您的个人喜好，除非您之前告诉过我。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 213, 'total_tokens': 234, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'agnes-2.0-flash', 'system_fingerprint': None, 'id': 'chatcmpl-4514da92-5f1a-4860-9f9e-4de9fbff6590', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ebc3e-319a-7962-9382-660563aa636b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 213, 'output_tokens': 21, 'total_tokens': 234, 'input_token_details': {}, 'output_token_details': {}})]}


## 记忆

<img src="./resources/memory.png" width="400" style="display:block; margin-left:0;">

### 现在我们开始设定记忆，可以看到我们是通过 THREAD 把连续的对话的历史都记录下来，可以看到从 A一直到 F 都有记忆，然后通过 THREAD 来调用，这样可以访问所有发生的历史。


In [6]:
from langgraph.checkpoint.memory import InMemorySaver  


agent = create_agent(
    model,
    checkpointer=InMemorySaver(),  
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content="你好，我是卫平，我喜欢蓝色")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [9]:
pprint(response)

{'messages': [HumanMessage(content='你好，我是卫平，我喜欢蓝色', additional_kwargs={}, response_metadata={}, id='346bd7b5-7267-4044-9a81-2b400c608269'),
              AIMessage(content='你好，卫平！很高兴认识你。蓝色确实是一种很迷人且宁静的颜色。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 217, 'total_tokens': 237, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'agnes-2.0-flash', 'system_fingerprint': None, 'id': 'chatcmpl-9c6da115-74ad-4908-9963-f6b0d179a209', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ebc0e-6064-7750-be78-3f952302cde1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 217, 'output_tokens': 20, 'total_tokens': 237, 'input_token_details': {}, 'output_token_details': {}})]}


In [8]:
question = HumanMessage(content="我喜欢什么颜色？")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pprint(response)

{'messages': [HumanMessage(content='你好，我是卫平，我喜欢蓝色', additional_kwargs={}, response_metadata={}, id='7cb33568-a855-44d6-ac06-3c3b75c72a74'),
              AIMessage(content='你好，卫平！很高兴认识你。蓝色确实是一种很令人喜爱的颜色，它通常象征着宁静、智慧和广阔。你最喜欢蓝色的哪些事物呢？比如天空、大海，还是某种特定的色调？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 217, 'total_tokens': 260, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'agnes-2.0-flash', 'system_fingerprint': 'vllm-0.21.0-tp2-52bef988', 'id': 'chatcmpl-8c9fac9936689f78', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ebc3e-5b13-7bf2-8dfe-8a6bb3bef24a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 217, 'output_tokens': 43, 'total_tokens': 260, 'input_token_details': {}, 'output_token_details': {}}),
              HumanMessage(content='我喜欢什么颜色？', additional_kwargs={}, response_metadata={}, id='6f7da15c-5dea-48bd-ab2f-29381c

<img src="./resources/mem_w_state.png" width="400" style="display:block; margin-left:0;">

#### 我们可以扩展记忆，，不仅包括对话历史，也可以增加我们定制化的新的数据项，例如用户的名字， 语言或任何其它数据项
